In [ ]:
import pandas as pd
import torch
from transformers import pipeline

country = ""
while country.lower() != "ch" and country.lower() != "us":
    print("Which country should be analyzed? (CH/US)")
    country = str(input())

df = pd.read_csv(f'{country}/songs_processed.csv')

print("Checking for Apple Silicon acceleration...")
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device.upper()}")

print("Downloading models...")

# -- SETUP PIPELINES --
sentiment_pipe = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest", truncation=True, max_length=512, device=device)

emotion_pipe = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", top_k=2, truncation=True, max_length=512, device=device)

zero_shot_pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=device)

# Categories for Zero-Shot Classification
categories = [
    "Romantic Love", "Heartbreak and Breakup", 
    "Wealth, Success and Flexing", "Social Issues and Protest", 
    "Escapism and Partying", "Mental Health and Struggles", 
    "Empowerment and Self-Confidence", "Nostalgia and Memories"
]


# -- HELPER METHODS --

def get_sentiment_and_score(text):
    if not text: return pd.Series([None, None])
    
    result = sentiment_pipe(text, truncation=True, max_length=512)
    return pd.Series([result[0]['label'], result[0]['score']])

def get_top2_emotions(text):
    if not text: return pd.Series([None, None, None, None])
    
    result = emotion_pipe(text, truncation=True, max_length=512)
    emotions = result[0] 
    
    # Returns Emotion 1, Score 1, Emotion 2 and Score 2
    if len(emotions) >= 2:
        return pd.Series([
            emotions[0]['label'], emotions[0]['score'], 
            emotions[1]['label'], emotions[1]['score']
        ])
    elif len(emotions) == 1:
         return pd.Series([emotions[0]['label'], emotions[0]['score'], None, None])
    else:
        return pd.Series([None, None, None, None])

def get_topic_and_score(text):
    if not text: return pd.Series([None, None])
    
    text_short = text[:1500] 
    result = zero_shot_pipe(text_short, candidate_labels=categories)
    
    return pd.Series([result['labels'][0], result['scores'][0]])


# -- DATA PROCESSING --
print("Starting analysis...")

print("Analyzing sentiment...")
df[['Sentiment', 'Sentiment_Score']] = df['Lyrics'].apply(get_sentiment_and_score)

print("Analyzing emotions...")
df[['Emotion_1', 'Emotion_1_Score', 'Emotion_2', 'Emotion_2_Score']] = df['Lyrics'].apply(get_top2_emotions)

print("Analyzing theme...")
df[['Topic', 'Topic_Score']] = df['Lyrics'].apply(get_topic_and_score)

print("Analysis done!")

df.to_csv(f'{country}/songs_analyzed.csv', index=False)

Which country should be analyzed? (CH/US)
Checking for Apple Silicon acceleration...
Using device: MPS


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Starting analysis...
Analyzing sentiment...
Analyzing emotions...
Analyzing theme...
Analysis done!
